# 05 — LightGBM

Primary model. Gradient boosting on tabular data consistently outperforms
Random Forest in accuracy and is significantly faster to train, which makes
a proper hyperparameter search feasible.

Additionally, LightGBM has native SHAP support, which lets us explain
individual predictions — important for a project like this where knowing
*why* a station is predicted to be full is as valuable as the number.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap

import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint, uniform

from src.utils.config import PATHS, FEATURES, TARGET, SPLIT_DATE, RANDOM_STATE, DATASET_FILE
from src.models.evaluate import (
    regression_metrics, plot_predictions, plot_residuals,
    plot_feature_importance, plot_error_by_group
)

print('Libraries loaded OK')

---
## 1. Load and split

In [ ]:
df = pd.read_parquet(DATASET_FILE)
df['datetime_hour'] = pd.to_datetime(df['datetime_hour'])

train = df[df['datetime_hour'] < SPLIT_DATE]
test  = df[df['datetime_hour'] >= SPLIT_DATE]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

avg_capacity = float(df['capacity'].drop_duplicates().mean())
print(f'Train: {len(train):,}  Test: {len(test):,}')

---
## 2. Hyperparameter search

In [ ]:
tscv = TimeSeriesSplit(n_splits=3, gap=168)

param_dist = {
    # Number of boosting rounds. LightGBM adds one tree per round to correct
    # the residuals of the previous ensemble. More rounds = finer corrections,
    # but too many will overfit — that's what learning_rate and regularization control.
    'n_estimators':    randint(200, 800),

    # Max number of leaves per tree. LightGBM grows leaf-wise (best-first) rather
    # than depth-wise, so num_leaves is the primary complexity knob. More leaves
    # = more complex patterns captured, but also more risk of overfitting.
    # Rule of thumb: num_leaves < 2^max_depth.
    'num_leaves':      randint(31, 127),

    # -1 means no depth limit — num_leaves controls complexity instead.
    # Setting a cap adds an extra safety net against very deep paths.
    'max_depth':       [-1, 8, 12, 16],

    # Shrinks each tree's contribution. Lower rate = the model learns more slowly
    # but generalises better, provided n_estimators is high enough to compensate.
    'learning_rate':   uniform(0.03, 0.15),

    # Row subsampling per tree (bagging). Using only a fraction of rows per round
    # introduces randomness that reduces overfitting, similar to Random Forest.
    'subsample':       uniform(0.7, 0.3),

    # Feature subsampling per tree. Same idea as subsample but applied to columns.
    # Particularly useful when some features are strongly correlated.
    'colsample_bytree':uniform(0.6, 0.4),

    # Min number of samples in a leaf. One of the most effective regularization
    # params in LightGBM — prevents the model from memorising tiny subgroups.
    'min_child_samples': randint(20, 100),

    # L1 regularization on leaf weights — pushes small weights to zero (sparse model).
    'reg_alpha':       uniform(0, 0.1),

    # L2 regularization on leaf weights — penalises large weights (smoother model).
    'reg_lambda':      uniform(0, 0.1),
}

lgbm = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)

search = RandomizedSearchCV(
    lgbm,
    param_distributions=param_dist,
    # 40 iterations since LightGBM is fast enough to afford a wider search.
    n_iter=40,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE,
    verbose=2,
    # n_jobs=1 here because LightGBM already parallelises internally via n_jobs=-1.
    # Running outer and inner parallelism simultaneously often hurts performance.
    n_jobs=1,
)

search.fit(X_train, y_train)
print(f'Best CV RMSE: {-search.best_score_:.4f}')
print(f'Best params:  {search.best_params_}')

---
## 3. Evaluate best model

In [ ]:
best_lgbm = search.best_estimator_

y_pred_train = best_lgbm.predict(X_train)
y_pred_test  = best_lgbm.predict(X_test)

print('=== LIGHTGBM ===')
m_train = regression_metrics(y_train, y_pred_train, 'Train', avg_capacity)
print()
m_test  = regression_metrics(y_test,  y_pred_test,  'Test',  avg_capacity)

---
## 4. RMSE by horizon

In [ ]:
from sklearn.metrics import mean_squared_error

rows = []
for h in sorted(test['horizon_hours'].unique()):
    mask = test['horizon_hours'] == h
    rmse = float(np.sqrt(mean_squared_error(y_test[mask], y_pred_test[mask])))
    rows.append({'horizon_hours': h, 'rmse': rmse})

df_h = pd.DataFrame(rows)
print(df_h.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_h['horizon_hours'], df_h['rmse'], marker='o', color='steelblue')
ax.set_xlabel('Horizon (hours ahead)')
ax.set_ylabel('RMSE')
ax.set_title('LightGBM — RMSE by prediction horizon')
plt.tight_layout()
plt.savefig(PATHS['figures'] / '05_lgbm_rmse_by_horizon.png', dpi=150)
plt.show()

---
## 5. Error analysis by distance to beach

Stations near the beach show the most extreme occupancy swings.
Checking whether the model handles them well is more informative than
looking at the global RMSE alone.

In [ ]:
test_eval = test.copy()
test_eval['y_pred'] = y_pred_test
test_eval['sq_err'] = (test_eval[TARGET] - test_eval['y_pred']) ** 2

test_eval['dist_group'] = pd.cut(
    test_eval['dist_beach'],
    bins=[0, 0.5, 1.5, 3, 100],
    labels=['<500m', '500m-1.5km', '1.5-3km', '>3km']
)

rmse_by_dist = (
    test_eval.groupby('dist_group')['sq_err']
    .mean().apply(np.sqrt)
    .reset_index()
    .rename(columns={'sq_err': 'rmse', 'dist_group': 'group'})
)

fig = plot_error_by_group(rmse_by_dist.rename(columns={'group': 'dist_group'}),
                          group_col='dist_group',
                          title='LightGBM — RMSE by distance to beach')
fig.savefig(PATHS['figures'] / '05_lgbm_rmse_by_dist.png', dpi=150)
plt.show()

---
## 6. SHAP — global feature importance

In [ ]:
# Use a sample to keep computation time reasonable
sample_idx = X_test.sample(5000, random_state=RANDOM_STATE).index
X_sample   = X_test.loc[sample_idx]

explainer   = shap.TreeExplainer(best_lgbm)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample, show=False)
plt.tight_layout()
plt.savefig(PATHS['figures'] / '05_lgbm_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. SHAP — single prediction example

This is the kind of explanation that would go into the deployment API response:
why did the model predict this specific occupancy for this specific station?

In [ ]:
# Pick one row and explain it
row_idx = 0
shap.waterfall_plot(
    shap.Explanation(values=shap_values[row_idx],
                     base_values=explainer.expected_value,
                     data=X_sample.iloc[row_idx],
                     feature_names=FEATURES),
    show=False
)
plt.tight_layout()
plt.savefig(PATHS['figures'] / '05_lgbm_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Prediction plots

In [ ]:
fig = plot_predictions(y_test.values, y_pred_test,
                       rmse=m_test['rmse'], r2=m_test['r2'],
                       title='LightGBM — Predicted vs Actual')
fig.savefig(PATHS['figures'] / '05_lgbm_predictions.png', dpi=150)
plt.show()

fig = plot_residuals(y_test.values, y_pred_test, title='LightGBM — Residuals')
fig.savefig(PATHS['figures'] / '05_lgbm_residuals.png', dpi=150)
plt.show()

---
## 9. Save model

In [ ]:
model_path = PATHS['models'] / 'lightgbm.joblib'
joblib.dump(best_lgbm, model_path)
print(f'Model saved to {model_path}')